# Imports

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.feature_engineering import create_features
from src.preprocessing import create_preprocessing_pipeline
from src.models import create_model
from src.evaluation import evaluate_model
from src.predict import create_submission

import pandas as pd

from sklearn.model_selection import train_test_split

# Load Data

In [3]:
train = pd.read_csv("../data/train.csv", index_col="PassengerId")
test = pd.read_csv("../data/test.csv", index_col="PassengerId")

# Feature Engineering

In [4]:
train = create_features(train)
test = create_features(test)

# Create X and y

In [5]:
X = train.drop("Survived", axis=1)

y = train["Survived"]

X_submission = test.copy()

# Train/Test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Features

In [7]:
numerical_features = X_train.select_dtypes(
    include="number"
).columns

categorical_features = X_train.select_dtypes(
    exclude="number"
).columns

# Pipeline

In [8]:
preprocess_pipeline = create_preprocessing_pipeline(
    numerical_features,
    categorical_features
)

# Transform

In [9]:
X_train_ready = preprocess_pipeline.fit_transform(X_train)

X_test_ready = preprocess_pipeline.transform(X_test)

X_submission_ready = preprocess_pipeline.transform(X_submission)

# Model

In [10]:
model = create_model()

model.fit(
    X_train_ready,
    y_train
)

CatBoostClassifier(border_count=16, depth=4, iterations=250, l2_leaf_reg=2, learning_rate=0.01, random_state=42, verbose=False)

# Evaluation

In [11]:
evaluate_model(
    model,
    X_test_ready,
    y_test
)

Accuracy: 0.8156

              precision    recall  f1-score   support

           0       0.83      0.87      0.85       110
           1       0.78      0.72      0.75        69

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.82      0.81       179


[[96 14]
 [19 50]]


# Submission

In [12]:
submission = create_submission(
    model,
    X_submission_ready,
    X_submission.index,
    "../submissions/submission.csv"
)

In [13]:
submission.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
